Easy -- Reshape round-trip. Take a tensor of shape (2, 3, 4). Reshape it to (6, 4), then to (24,), then back to (2, 3, 4). Verify element order is preserved at each step by printing the flat data.

In [1]:
import numpy as np
a=np.arange(24).reshape(2,3,4)
print("original:",a.shape)
print("flat:",a.flatten())

original: (2, 3, 4)
flat: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23]


In [2]:
a

array([[[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]],

       [[12, 13, 14, 15],
        [16, 17, 18, 19],
        [20, 21, 22, 23]]])

In [3]:
b=a.reshape(6,4)
print("(6,4):", b.shape)
print("flat:", b.flatten())

(6,4): (6, 4)
flat: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23]


In [4]:
b

array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11],
       [12, 13, 14, 15],
       [16, 17, 18, 19],
       [20, 21, 22, 23]])

In [5]:
c = a.reshape(24)
print("(24,):", c.shape)
print("flat:", c.flatten())

(24,): (24,)
flat: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23]


In [6]:
c

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23])

In [9]:
d = c.reshape(2, 3, 4)
print("back to (2,3,4):", d.shape)
print("flat:", d.flatten())

back to (2,3,4): (2, 3, 4)
flat: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23]


In [10]:
d

array([[[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]],

       [[12, 13, 14, 15],
        [16, 17, 18, 19],
        [20, 21, 22, 23]]])

In [11]:
print(np.array_equal(a, d))              # True
print(np.array_equal(a.flatten(), d.flatten()))  # True

True
True


Medium -- Implement broadcasting. Extend the Tensor class with a broadcast_to(shape) method that expands dimensions of size 1 to match a target shape. Then modify _elementwise_op to automatically broadcast before operating. Test with shapes (3, 1) and (1, 4) producing (3, 4).

Hard -- Build einsum from scratch. Implement a basic einsum(subscripts, *tensors) function that handles at least: dot product (i,i->), matrix multiply (ij,jk->ik), outer product (i,j->ij), and transpose (ij->ji). Parse the subscript string, identify contracted indices, and loop over all index combinations. Compare your results against np.einsum.

In [12]:
import numpy as np
from itertools import product

def einsum(subscripts, *tensors):
    # ---------- 1. Parse ----------
    if '->' in subscripts:
        lhs, rhs = subscripts.split('->')
        output_labels = rhs.strip()
    else:
        # implicit mode: letters appearing exactly once, sorted
        lhs = subscripts
        counts = {}
        for ch in lhs.replace(',', ''):
            counts[ch] = counts.get(ch, 0) + 1
        output_labels = ''.join(sorted(c for c, n in counts.items() if n == 1))

    input_labels = [s.strip() for s in lhs.split(',')]

    # ---------- 2. Map letter -> size ----------
    dims = {}
    for labels, tensor in zip(input_labels, tensors):
        for letter, size in zip(labels, tensor.shape):
            if letter in dims and dims[letter] != size:
                raise ValueError(f"size mismatch for '{letter}'")
            dims[letter] = size

    # ---------- 3. All letters that need iterating ----------
    # Order matters only for consistency; use insertion order
    all_letters = list(dims.keys())
    ranges = [range(dims[l]) for l in all_letters]

    # ---------- 4. Accumulate ----------
    # Output shape
    out_shape = tuple(dims[l] for l in output_labels)
    result = np.zeros(out_shape, dtype=float)

    # Precompute: which axis of each tensor each letter maps to
    letter_to_axis = []
    for labels in input_labels:
        letter_to_axis.append({l: i for i, l in enumerate(labels)})

    # Output index map
    out_index = {l: i for i, l in enumerate(output_labels)}

    # Loop over every combination of all letters
    for combo in product(*ranges):
        idx = dict(zip(all_letters, combo))

        # Multiply the corresponding elements of all tensors
        prod_val = 1.0
        for labels, tensor, axmap in zip(input_labels, tensors, letter_to_axis):
            # Build the index tuple for this tensor
            tensor_idx = tuple(idx[l] for l in labels)
            prod_val *= tensor[tensor_idx]

        # Decide where to put it in the output
        if output_labels == '':
            result[...] += prod_val
        else:
            out_idx = tuple(idx[l] for l in output_labels)
            result[out_idx] += prod_val

    return result

In [13]:
np.random.seed(0)

# 1. Dot product: i,i->
a = np.random.rand(5)
b = np.random.rand(5)
print("dot:",
      np.allclose(einsum('i,i->', a, b), np.einsum('i,i->', a, b)))
# True

# 2. Matrix multiply: ij,jk->ik
A = np.random.rand(3, 4)
B = np.random.rand(4, 5)
print("matmul:",
      np.allclose(einsum('ij,jk->ik', A, B), np.einsum('ij,jk->ik', A, B)))
# True

# 3. Outer product: i,j->ij
x = np.random.rand(3)
y = np.random.rand(4)
print("outer:",
      np.allclose(einsum('i,j->ij', x, y), np.einsum('i,j->ij', x, y)))
# True

# 4. Transpose: ij->ji
M = np.random.rand(2, 5)
print("transpose:",
      np.allclose(einsum('ij->ji', M), np.einsum('ij->ji', M)))
# True

# 5. Bonus: trace (ii->) — works for free
S = np.random.rand(4, 4)
print("trace:",
      np.allclose(einsum('ii->', S), np.einsum('ii->', S)))
# True

dot: True
matmul: True
outer: True
transpose: True
trace: True
